# 🧬 HematoScan — End-to-End ML Pipeline

**Blood Cancer Detection System** — Classifies CBC blood test and blood smear image data into:
- **Normal** — Healthy blood profile
- **Leukemia** — Acute/Chronic Leukemia (ALL, AML, CLL, CML)
- **Lymphoma** — Hodgkin / Non-Hodgkin Lymphoma
- **Myeloma** — Multiple Myeloma

This notebook walks through the full pipeline: data generation → model training → evaluation → prediction.

---

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 1.  Setup & Imports
# ─────────────────────────────────────────────────────────────────
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
np.random.seed(42)

# Add backend root to path so we can import our modules
BACKEND_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if BACKEND_ROOT not in sys.path:
    sys.path.insert(0, BACKEND_ROOT)

from app.ml.data import (
    CLASS_LABELS, CBC_FEATURES, IMAGE_FEATURES,
    CBC_CLASS_PATTERNS, IMAGE_CLASS_PATTERNS,
    generate_cbc_samples, generate_image_samples,
    export_all_datasets, load_csv_dataset, dataset_stats,
)
from app.ml.train import train_blood_test_model, train_image_model
from app.models.detection import BloodTestData

print(f"🎯 Classes: {CLASS_LABELS}")
print(f"📊 CBC features ({len(CBC_FEATURES)}): {CBC_FEATURES}")
print(f"🖼️  Image features ({len(IMAGE_FEATURES)}): {IMAGE_FEATURES}")
print()
print("✅ All imports successful")

---
## 📊 2.  Data Exploration

Let's visualize the synthetic data patterns for each cancer type.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2a.  Medical Pattern Overview
# ─────────────────────────────────────────────────────────────────
print("=" * 70)
print(f"{'Class':<12s} {'Key CBC Indicators':<50s}")
print("=" * 70)

for label in CLASS_LABELS:
    p = CBC_CLASS_PATTERNS[label]
    print(f"{label:<12s} {p['description']}")

print()
print("=" * 70)
print(f"{'Class':<12s} {'Key Image Indicators':<50s}")
print("=" * 70)
for label in CLASS_LABELS:
    p = IMAGE_CLASS_PATTERNS[label]
    print(f"{label:<12s} {p['description']}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2b.  Generate & Visualize CBC Data
# ─────────────────────────────────────────────────────────────────
df_cbc = generate_cbc_samples(n_per_class=500)
print(f"CBC dataset: {df_cbc.shape[0]} samples, {len(CBC_FEATURES)} features")
df_cbc.head()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2c.  CBC Feature Distributions by Class
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(CBC_FEATURES):
    for cls_name in CLASS_LABELS:
        subset = df_cbc[df_cbc['target'] == cls_name][feat]
        sns.kdeplot(subset, label=cls_name, ax=axes[i], fill=True, alpha=0.3)
    axes[i].set_title(feat, fontsize=11)
    axes[i].legend(fontsize=7)

plt.tight_layout()
plt.suptitle('CBC Feature Distributions by Cancer Type', fontsize=14, y=1.02)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2d.  Key Discriminating Features (Box Plots)
# ─────────────────────────────────────────────────────────────────
key_features = ['wbc', 'hemoglobin', 'platelets', 'lymphocytes', 'blast_cells']
fig, axes = plt.subplots(1, 5, figsize=(20, 5))

for i, feat in enumerate(key_features):
    sns.boxplot(data=df_cbc, x='target', y=feat, ax=axes[i], palette='Set2')
    axes[i].set_title(feat.upper(), fontsize=12)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Key Blood Markers Across Cancer Types', fontsize=14, y=1.02)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2e.  Image Feature Distributions
# ─────────────────────────────────────────────────────────────────
df_img = generate_image_samples(n_per_class=500)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(IMAGE_FEATURES):
    for cls_name in CLASS_LABELS:
        subset = df_img[df_img['target'] == cls_name][feat]
        sns.kdeplot(subset, label=cls_name, ax=axes[i], fill=True, alpha=0.3)
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=7)

plt.tight_layout()
plt.suptitle('Image Feature Distributions by Cancer Type', fontsize=14, y=1.02)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 2f.  Correlation Heatmap
# ─────────────────────────────────────────────────────────────────
# Encode labels for correlation analysis
label_map = {l: i for i, l in enumerate(CLASS_LABELS)}
df_cbc_corr = df_cbc.copy()
df_cbc_corr['target_encoded'] = df_cbc_corr['target'].map(label_map)

plt.figure(figsize=(12, 8))
corr = df_cbc_corr[CBC_FEATURES + ['target_encoded']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('CBC Feature Correlation Matrix with Target', fontsize=14)
plt.tight_layout()
plt.show()

# Feature importance ranking (absolute correlation with target)
importance = corr['target_encoded'].abs().sort_values(ascending=False)
print("\n📈 Top features correlated with cancer type:")
for feat, val in importance.items():
    if feat != 'target_encoded':
        print(f"   {feat:<14s}: {val:.3f}")

---
## 🧠 3.  Model Training — Scikit-Learn


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3a.  Prepare Training Data
# ─────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

N_SAMPLES = 1000  # per class

# Blood test data
df_cbc_full = generate_cbc_samples(n_per_class=N_SAMPLES)
X_cbc = df_cbc_full[CBC_FEATURES]
y_cbc = df_cbc_full['target'].map({'Normal': 0, 'Leukemia': 1, 'Lymphoma': 2, 'Myeloma': 3})

X_cbc_train, X_cbc_test, y_cbc_train, y_cbc_test = train_test_split(
    X_cbc, y_cbc, test_size=0.2, random_state=42, stratify=y_cbc
)

# Image feature data
df_img_full = generate_image_samples(n_per_class=N_SAMPLES)
X_img = df_img_full[IMAGE_FEATURES]
y_img = df_img_full['target'].map({'Normal': 0, 'Leukemia': 1, 'Lymphoma': 2, 'Myeloma': 3})

X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42, stratify=y_img
)

print(f"📊 CBC Training:   {X_cbc_train.shape[0]} samples, {X_cbc_train.shape[1]} features")
print(f"📊 CBC Test:       {X_cbc_test.shape[0]} samples")
print(f"🖼️  Image Training: {X_img_train.shape[0]} samples, {X_img_train.shape[1]} features")
print(f"🖼️  Image Test:     {X_img_test.shape[0]} samples")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3b.  Train Blood Test Model (Random Forest)
# ─────────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

blood_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=5,
        min_samples_leaf=2, class_weight='balanced',
        random_state=42, n_jobs=-1,
    )),
])

print("🔬 Training Blood Test Model (Random Forest)...")
blood_pipeline.fit(X_cbc_train, y_cbc_train)

y_cbc_pred = blood_pipeline.predict(X_cbc_test)
cbc_acc = (y_cbc_pred == y_cbc_test).mean()
print(f"✅ Accuracy: {cbc_acc:.3f}\n")
print(classification_report(y_cbc_test, y_cbc_pred, target_names=CLASS_LABELS))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3c.  CBC Confusion Matrix
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_cbc_test, y_cbc_pred,
    display_labels=CLASS_LABELS,
    cmap='Blues', ax=ax[0],
    colorbar=False,
)
ax[0].set_title(f'Blood Test (CBC) — Accuracy: {cbc_acc:.3f}', fontsize=12)
ax[0].tick_params(axis='x', rotation=30)

# Feature importance
rf = blood_pipeline.named_steps['classifier']
importances = pd.DataFrame({
    'feature': CBC_FEATURES,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

ax[1].barh(importances['feature'], importances['importance'], color='steelblue')
ax[1].set_title('Feature Importance (Random Forest)', fontsize=12)
ax[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3d.  Train Image Model (Gradient Boosting)
# ─────────────────────────────────────────────────────────────────
from sklearn.ensemble import GradientBoostingClassifier

img_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', GradientBoostingClassifier(
        n_estimators=150, max_depth=8,
        min_samples_split=5, learning_rate=0.1,
        random_state=42,
    )),
])

print("🖼️  Training Image Model (Gradient Boosting)...")
img_pipeline.fit(X_img_train, y_img_train)

y_img_pred = img_pipeline.predict(X_img_test)
img_acc = (y_img_pred == y_img_test).mean()
print(f"✅ Accuracy: {img_acc:.3f}\n")
print(classification_report(y_img_test, y_img_pred, target_names=CLASS_LABELS))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3e.  Image Model Confusion Matrix
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_img_test, y_img_pred,
    display_labels=CLASS_LABELS,
    cmap='Greens', ax=ax[0],
    colorbar=False,
)
ax[0].set_title(f'Image Features — Accuracy: {img_acc:.3f}', fontsize=12)
ax[0].tick_params(axis='x', rotation=30)

# Feature importance
gb = img_pipeline.named_steps['classifier']
importances = pd.DataFrame({
    'feature': IMAGE_FEATURES,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=True)

ax[1].barh(importances['feature'], importances['importance'], color='seagreen')
ax[1].set_title('Feature Importance (Gradient Boosting)', fontsize=12)
ax[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 3f.  Save Models to Disk
# ─────────────────────────────────────────────────────────────────
import joblib

MODELS_DIR = os.path.join(BACKEND_ROOT, 'app', 'ml', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

joblib.dump(blood_pipeline, os.path.join(MODELS_DIR, 'blood_test_model.pkl'))
joblib.dump(img_pipeline, os.path.join(MODELS_DIR, 'image_model.pkl'))
print(f"💾 Models saved to: {MODELS_DIR}")
print(f"   - blood_test_model.pkl")
print(f"   - image_model.pkl")

---
## 🧠 4.  CNN Training (TensorFlow/Keras)

Trains on synthetic blood smear images. Replace with real images for production accuracy.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4a.  Generate Synthetic Images (if not already present)
# ─────────────────────────────────────────────────────────────────
IMG_DIR = os.path.join(BACKEND_ROOT, 'data', 'images')
os.makedirs(IMG_DIR, exist_ok=True)

# Check if images exist
from pathlib import Path
existing = list(Path(IMG_DIR).rglob('*.png'))
print(f"Existing images in {IMG_DIR}: {len(existing)}")

if len(existing) < 400:
    print("\n⚙️  Generating synthetic images for CNN training...")
    from app.ml.generate_synthetic_images import generate_dataset
    generate_dataset(IMG_DIR, samples_per_class=300)
else:
    print("✅ Enough images already exist. Skipping generation.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4b.  Show Sample Images from Each Class
# ─────────────────────────────────────────────────────────────────
from PIL import Image

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, cls_name in enumerate(CLASS_LABELS):
    cls_dir = os.path.join(IMG_DIR, cls_name)
    if not os.path.exists(cls_dir):
        print(f"⚠️  No images for {cls_name}")
        continue
    files = sorted(os.listdir(cls_dir))
    if not files:
        continue
    img_path = os.path.join(cls_dir, files[0])
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].set_title(f'{cls_name}\n({os.path.getsize(img_path)//1024} KB)', fontsize=11, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Sample Synthetic Blood Smear Images', fontsize=14, y=1.02)
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4c.  Train CNN
# ─────────────────────────────────────────────────────────────────
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks
    
    print(f"🧠 TensorFlow: {tf.__version__}")
    print(f"   GPU: {'✅ Available' if len(tf.config.list_physical_devices('GPU')) > 0 else '⚠️  CPU only'}")
    
    IMG_SIZE = 224
    
    # Data augmentation
    datagen = keras.preprocessing.image.ImageDataGenerator(
        rescale=1.0/255,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.15,
        horizontal_flip=True,
        validation_split=0.2,
    )
    
    train_gen = datagen.flow_from_directory(
        IMG_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=32,
        class_mode='categorical',
        classes=CLASS_LABELS,
        subset='training',
        shuffle=True,
    )
    
    val_gen = datagen.flow_from_directory(
        IMG_DIR,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=32,
        class_mode='categorical',
        classes=CLASS_LABELS,
        subset='validation',
        shuffle=False,
    )
    
    print(f"\n   Training samples: {train_gen.samples}")
    print(f"   Validation samples: {val_gen.samples}")
    
    # Build model
    USE_PRETRAINED = len(existing) < 2000  # Use transfer learning for small datasets
    
    if USE_PRETRAINED:
        print("\n🔄 Using MobileNetV2 transfer learning...")
        base = keras.applications.MobileNetV2(
            input_shape=(IMG_SIZE, IMG_SIZE, 3),
            include_top=False,
            weights='imagenet',
        )
        base.trainable = False
        
        inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = keras.applications.mobilenet_v2.preprocess_input(inputs)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        outputs = layers.Dense(4, activation='softmax')(x)
        model = keras.Model(inputs=inputs, outputs=outputs, name='hematoscan_cnn_pretrained')
        lr = 1e-4
    else:
        print("\n🔧 Building custom CNN...")
        inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
        x = layers.Rescaling(1.0/255)(inputs)
        x = layers.Conv2D(32, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(64, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(128, 3, padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation='relu')(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.Dropout(0.3)(x)
        outputs = layers.Dense(4, activation='softmax')(x)
        model = keras.Model(inputs=inputs, outputs=outputs, name='hematoscan_cnn')
        lr = 1e-3
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')],
    )
    
    model.summary()
    
    # Callbacks
    CNN_MODEL_PATH = os.path.join(MODELS_DIR, 'cnn_image_model.keras')
    cb = [
        callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6, monitor='val_loss'),
        callbacks.ModelCheckpoint(CNN_MODEL_PATH, save_best_only=True, monitor='val_accuracy'),
    ]
    
    print(f"\n{'='*50}")
    print("🏋️  Starting CNN Training...")
    print(f"{'='*50}\n")
    
    EPOCHS = 30
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=cb,
        verbose=1,
    )
    
    print(f"\n✅ CNN training complete!")
    print(f"   Best val accuracy: {max(history.history['val_accuracy']):.3f}")
    print(f"   Best val AUC: {max(history.history['val_auc']):.3f}")
    
except ImportError as e:
    print(f"❌ TensorFlow not available: {e}")
    print("   Run: pip install tensorflow")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 4d.  CNN Training Curves
# ─────────────────────────────────────────────────────────────────
try:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train', lw=2)
    axes[0].plot(history.history['val_accuracy'], label='Validation', lw=2)
    axes[0].set_title('Accuracy', fontsize=12)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train', lw=2)
    axes[1].plot(history.history['val_loss'], label='Validation', lw=2)
    axes[1].set_title('Loss', fontsize=12)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # AUC
    if 'val_auc' in history.history:
        axes[2].plot(history.history['auc'], label='Train', lw=2)
        axes[2].plot(history.history['val_auc'], label='Validation', lw=2)
        axes[2].set_title('AUC (Area Under ROC)', fontsize=12)
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('AUC')
        axes[2].legend()
        axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.suptitle('CNN Training History', fontsize=14, y=1.02)
    plt.show()
    
    print(f"📈 Best validation accuracy: {max(history.history['val_accuracy']):.3f}")
    
except NameError:
    print("⚠️  CNN training didn't run yet. Run section 4c first.")
except Exception as e:
    print(f"⚠️  Could not plot: {e}")

---
## 📈 5.  Model Evaluation & Comparison

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5a.  All Models Comparison
# ─────────────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, f1_score

results = []

# Blood test model
cbc_f1 = f1_score(y_cbc_test, y_cbc_pred, average='weighted')
cbc_proba = blood_pipeline.predict_proba(X_cbc_test)
cbc_auc = roc_auc_score(pd.get_dummies(y_cbc_test), cbc_proba, multi_class='ovr')
results.append({
    'Model': 'Random Forest (CBC)',
    'Accuracy': cbc_acc,
    'F1-Score': cbc_f1,
    'AUC (OVR)': cbc_auc,
})

# Image model
img_f1 = f1_score(y_img_test, y_img_pred, average='weighted')
img_proba = img_pipeline.predict_proba(X_img_test)
img_auc = roc_auc_score(pd.get_dummies(y_img_test), img_proba, multi_class='ovr')
results.append({
    'Model': 'Gradient Boosting (Image)',
    'Accuracy': img_acc,
    'F1-Score': img_f1,
    'AUC (OVR)': img_auc,
})

# CNN (if available)
try:
    cnn_acc = max(history.history['val_accuracy'])
    cnn_auc = max(history.history['val_auc'])
    results.append({
        'Model': 'CNN (Image)',
        'Accuracy': cnn_acc,
        'F1-Score': cnn_acc,  # approximate from val accuracy
        'AUC (OVR)': cnn_auc,
    })
except:
    pass

df_results = pd.DataFrame(results).round(4)
print("📊 Model Performance Comparison:")
print(df_results.to_string(index=False))

# Visual comparison
fig, ax = plt.subplots(figsize=(10, 4))
metrics = ['Accuracy', 'F1-Score', 'AUC (OVR)']
x = np.arange(len(metrics))
width = 0.25

for i, (_, row) enumerate(df_results.iterrows()):
    values = [row[m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=row['Model'])
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.2f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 5b.  Cross-Validation Scores
# ─────────────────────────────────────────────────────────────────
from sklearn.model_selection import cross_val_score

print("📊 5-Fold Cross-Validation:")
print("-" * 40)

for name, pipeline, X, y in [
    ('Random Forest (CBC)', blood_pipeline, X_cbc, y_cbc),
    ('Gradient Boosting (Image)', img_pipeline, X_img, y_img),
]:
    scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
    print(f"{name:<30s}: {scores.mean():.3f} ± {scores.std()*2:.3f}")
    print(f"{'':30s}  Scores: {[f'{s:.3f}' for s in scores]}")
    print()

---
## 🎯 6.  Prediction Demo

Let's make live predictions and see the full inference pipeline in action.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 6a.  Blood Test Prediction Demo
# ─────────────────────────────────────────────────────────────────
def predict_blood_demo(wbc, rbc, hemoglobin, platelets, neutrophils,
                       lymphocytes, monocytes, eosinophils, basophils, blast_cells):
    """Predict cancer type from CBC parameters."""
    features = pd.DataFrame([[
        wbc, rbc, hemoglobin, platelets, neutrophils,
        lymphocytes, monocytes, eosinophils, basophils, blast_cells
    ]], columns=CBC_FEATURES)
    
    class_idx = blood_pipeline.predict(features)[0]
    proba = blood_pipeline.predict_proba(features)[0]
    
    return {
        'prediction': CLASS_LABELS[class_idx],
        'confidence': proba[class_idx],
        'all_probs': {cls: proba[i] for i, cls in enumerate(CLASS_LABELS)},
    }


# Demo: Test each cancer type with typical values
demo_cases = [
    ('Healthy Patient', 7.5, 5.2, 14.5, 250, 60, 30, 5, 2.5, 0.5, 0.5),
    ('Leukemia Suspect', 28.0, 3.2, 8.0, 90, 25, 52, 5, 2.0, 0.8, 38.0),
    ('Lymphoma Suspect', 13.0, 4.5, 11.5, 190, 22, 58, 8, 4.0, 1.2, 3.0),
    ('Myeloma Suspect', 9.0, 3.6, 9.5, 120, 55, 28, 7, 2.5, 0.6, 2.0),
]

results = []
for name, *params in demo_cases:
    r = predict_blood_demo(*params)
    results.append({'Case': name, **r})

df_demo = pd.DataFrame(results)
print("🎯 Blood Test Predictions:")
for _, row in df_demo.iterrows():
    probs_str = ' | '.join(f'{k}: {v:.1%}' for k, v in row['all_probs'].items())
    print(f"\n{'─'*60}")
    print(f"  📋 {row['Case']}")
    print(f"  ➡️  Prediction: {row['prediction']}  (confidence: {row['confidence']:.1%})")
    print(f"  📊 All classes: {probs_str}")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 6b.  Image Prediction Demo (CNN)
# ─────────────────────────────────────────────────────────────────
def predict_image_demo(image_path):
    """Predict cancer type from a blood smear image."""
    if not os.path.exists(image_path):
        return {'error': f'File not found: {image_path}'}
    
    from PIL import Image
    
    # OpenCV + GB model
    from app.ml.data import extract_image_features
    features = extract_image_features(image_path)
    features_df = pd.DataFrame([features], columns=IMAGE_FEATURES)
    
    class_idx_gb = img_pipeline.predict(features_df)[0]
    proba_gb = img_pipeline.predict_proba(features_df)[0]
    
    # CNN (if available)
    cnn_result = None
    try:
        img = Image.open(image_path).convert("RGB").resize((224, 224))
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        cnn_probs = model.predict(img_array, verbose=0)[0]
        cnn_idx = int(np.argmax(cnn_probs))
        cnn_result = {
            'prediction': CLASS_LABELS[cnn_idx],
            'confidence': float(cnn_probs[cnn_idx]),
        }
    except Exception as e:
        cnn_result = {'error': str(e)}
    
    return {
        'GB_model': {
            'prediction': CLASS_LABELS[class_idx_gb],
            'confidence': float(proba_gb[class_idx_gb]),
            'all_probs': {cls: float(proba_gb[i]) for i, cls in enumerate(CLASS_LABELS)},
        },
        'CNN_model': cnn_result,
    }


# Test with one sample from each class
for cls_name in CLASS_LABELS:
    cls_dir = os.path.join(IMG_DIR, cls_name)
    if not os.path.exists(cls_dir):
        continue
    files = sorted(os.listdir(cls_dir))
    if not files:
        continue
    test_img = os.path.join(cls_dir, files[0])
    
    print(f"\n{'─'*60}")
    print(f"  🖼️  Testing image: {cls_name}/{files[0]}")
    
    result = predict_image_demo(test_img)
    
    if 'GB_model' in result:
        gb = result['GB_model']
        probs = ' | '.join(f'{k}: {v:.1%}' for k, v in gb['all_probs'].items())
        print(f"  📊 GB Model: {gb['prediction']} ({gb['confidence']:.1%})")
        print(f"     {probs}")
    
    if result.get('CNN_model'):
        cnn = result['CNN_model']
        print(f"  🧠 CNN:      {cnn['prediction']} ({cnn['confidence']:.1%})")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 6c.  Batch Prediction Visualization
# ─────────────────────────────────────────────────────────────────
n_test = min(50, len(list(Path(IMG_DIR).rglob('*.png'))))
test_images = []
for cls in CLASS_LABELS:
    cls_dir = os.path.join(IMG_DIR, cls)
    if os.path.exists(cls_dir):
        files = sorted(os.listdir(cls_dir))[:10]
        test_images.extend([(os.path.join(cls_dir, f), cls) for f in files])

# Predict on a grid of sample images
n_grid = min(len(test_images), 8)
fig, axes = plt.subplots(2, n_grid // 2, figsize=(3 * n_grid // 2, 6))
if n_grid <= 2:
    axes = [axes]
else:
    axes = axes.flatten()

for i in range(n_grid):
    if i >= len(test_images):
        break
    img_path, true_label = test_images[i]
    
    # Predict with GB model
    from app.ml.data import extract_image_features
    features = extract_image_features(img_path)
    pred_idx = img_pipeline.predict(pd.DataFrame([features], columns=IMAGE_FEATURES))[0]
    pred_label = CLASS_LABELS[pred_idx]
    
    img = Image.open(img_path)
    axes[i].imshow(img)
    color = 'green' if pred_label == true_label else 'red'
    axes[i].set_title(f'True: {true_label}\nPred: {pred_label}', fontsize=9, color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.suptitle('Sample Predictions (🟢 Correct  🔴 Incorrect)', fontsize=13, y=1.02)
plt.show()

---
## 🌐 7.  Download Real Data from Kaggle

To improve accuracy with real data, download datasets from Kaggle:

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 7.  Kaggle Setup & Download
# ─────────────────────────────────────────────────────────────────
import subprocess

def check_kaggle():
    """Check if Kaggle CLI is authenticated."""
    result = subprocess.run(
        ['kaggle', 'datasets', 'list', '--page-size', '1'],
        capture_output=True, text=True, timeout=15,
    )
    return result.returncode == 0

if check_kaggle():
    print("✅ Kaggle API is authenticated and ready!")
    print("\n   To download datasets, run cells below.")
else:
    print("❌ Kaggle API not authenticated.")
    print("\n   Setup steps:")
    print("   1. Go to https://www.kaggle.com/settings")
    print("   2. Scroll to API section → 'Create New Token'")
    print("   3. Move kaggle.json to ~/.kaggle/kaggle.json")
    print("   4. Run: chmod 600 ~/.kaggle/kaggle.json")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 7b.  Download & Organize Datasets (uncomment to run)
# ─────────────────────────────────────────────────────────────────
# # Download leukemia dataset
# !kaggle datasets download -d andrewmvd/leukemia-classification --unzip -p ./data/raw_kaggle/leukemia

# # Download lymphoma dataset
# !kaggle datasets download -d andrewmvd/malignant-lymphoma-classification --unzip -p ./data/raw_kaggle/lymphoma

# # Organize into 4-class structure
# !python -m app.ml.download_datasets --organize

print("📋 Uncomment the cells above to download real data from Kaggle.")
print("   Then re-run cell 6c to train on real images!")

---
## ✅ Summary

| Component | Model | Status |
|-----------|-------|--------|
| Blood Test (CBC) | Random Forest | ✅ Trained & Saved |
| Image Features | Gradient Boosting | ✅ Trained & Saved |
| Deep Learning (Images) | CNN (TensorFlow) | ✅ Trained & Saved |

**Next steps for production accuracy:**
1. Download real Kaggle datasets (cell 7b)
2. Re-train on real images
3. Fine-tune hyperparameters
4. Deploy to production using the FastAPI backend

In [ ]:
print("🎉 HematoScan pipeline complete!")
print(f"")
print(f"📁 Models saved to: {MODELS_DIR}")
print(f"📁 Data saved to:   {os.path.join(BACKEND_ROOT, 'data')}")
print(f"")
print(f"🚀 To start the app:")
print(f"   cd backend && source .venv3.11/bin/activate && uvicorn app.main:app --reload")